## Load Image location metadata

In [17]:
import pandas as pd
import sys

# Load the metadata

select_split = 'test'

path = "../chestx-ray/VinDr-PCXR"

if select_split == 'train':
    df = pd.read_csv(path + "/train/image_labels_train.csv")
elif select_split == 'test':
    df = pd.read_csv(path + "/test/image_labels_test.csv")

class_dist = df.sum()

print("class distance:", class_dist)

class distance: image_id                    d7e71a052a753c3f2f3e317d60177bece64d9421e78c82...
rad_ID                      R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3R3...
No finding                                                              907.0
Bronchitis                                                              174.0
Brocho-pneumonia                                                         84.0
Other disease                                                            77.0
Bronchiolitis                                                            90.0
Situs inversus                                                            2.0
Pneumonia                                                                89.0
Pleuro-pneumonia                                                          0.0
Diagphramatic hernia                                                      0.0
Tuberculosis                                                              1.0
Congenital emphysema                            

## Create download txt

In [18]:
# Tuberculosis
pathologies = ["Pneumonia"]

df_download = df[df[pathologies].sum(axis=1) > 0]
df_download = df


print(len(df_download))
 

1397


In [19]:
# Create the list of URLs
if select_split == 'train':
    base_url = "https://physionet.org/files/vindr-pcxr/1.0.0/train/"
elif select_split == 'test':
    base_url = "https://physionet.org/files/vindr-pcxr/1.0.0/test/"

with open(path + '/download_list_full.txt', 'w') as f:
    for img_id in df_download['image_id']:
        f.write(f"{base_url}{img_id}.dicom\n")

print(f"Done! Created 'download_list_full.txt' with {len(df_download)} targeted URLs.")

Done! Created 'download_list_full.txt' with 1397 targeted URLs.


## Execute Downloading of download list

In [20]:
import os
import requests
from tqdm import tqdm

# Deine Daten
USERNAME = 'kallepalle'
PASSWORD = 'Appl1edD33p' # Dein echtes Passwort nutzen
LIST_FILE = '../chestx-ray/VinDr-PCXR/download_list_full.txt'

if select_split == 'train':
    OUTPUT_DIR = f'../chestx-ray/VinDr-PCXR/train'
elif select_split == 'test':
    OUTPUT_DIR = f'../chestx-ray/VinDr-PCXR/test'

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

session = requests.Session()

# 1. Login-Seite aufrufen, um den CSRF-Token zu erhalten
login_url = "https://physionet.org/login/"
get_response = session.get(login_url)
csrftoken = session.cookies.get('csrftoken')

# 2. Login-Daten mit CSRF-Token vorbereiten
login_data = {
    'username': USERNAME,
    'password': PASSWORD,
    'csrfmiddlewaretoken': csrftoken,
    'next': '/content/vindr-pcxr/1.0.0/'
}

# 3. Einloggen (Referer-Header ist oft wichtig)
session.post(login_url, data=login_data, headers={'Referer': login_url})

# 4. Download-Liste einlesen
with open(LIST_FILE, 'r') as f:
    urls = [line.strip() for line in f if line.strip()]

print(f"Starte sicheren Download von {len(urls)} Dateien...")

for url in tqdm(urls):
    filename = os.path.join(OUTPUT_DIR, os.path.basename(url))
    
    # Prüfen, ob Datei schon existiert (spart Zeit bei Abbruch)
    if os.path.exists(filename):
        continue

    response = session.get(url, stream=True)
    
    if response.status_code == 200:
        with open(filename, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024*1024): # 1MB Chunks
                f.write(chunk)
    elif response.status_code == 403:
        print(f"\nFehler 403 bei {url}. Login fehlgeschlagen oder DUA nicht aktiv.")
        break 
    else:
        print(f"\nFehler {response.status_code} bei {url}")

print("\nFertig!")

Starte sicheren Download von 1397 Dateien...


100%|██████████| 1397/1397 [00:20<00:00, 67.73it/s] 


Fertig!
